# Proyecto Data Science - Parte Proyecto Final: Análisis de Datos de Publicidad en Redes Sociales

**Autor:** Xavier Gutierrez  
**Fecha:** Octubre 2025  
**Curso:** CoderHouse - Data Science I  
**Comisión:** 77695

**Fuente de datos:** [Social Media Advertisement Performance Dataset](https://www.kaggle.com/datasets/alperenmyung/social-media-advertisement-performance?select=ad_events.csv)

**Nota sobre la fuente:** El archivo `ad_campaign_db.sqlite` incluido en el dataset de Kaggle fue fundamental para entender las relaciones entre las tablas, lo que facilitó la realización de los merge entre los diferentes archivos CSV.

---

## 1. Abstract

Este proyecto analiza el desempeño de campañas publicitarias combinando los datasets `ad_events`, `ads`, `campaigns` y `users`. La Parte I integró las fuentes y ejecutó un análisis exploratorio para detectar patrones de interacción por campaña, plataforma, franja horaria y audiencia. En esta Parte III se incorpora un flujo de Machine Learning orientado a estimar el CTR (click through rate) por anuncio mediante un árbol de regresión, con etapas de preparación de datos, selección de variables y evaluación del modelo. Los hallazgos combinan métricas descriptivas y predictivas que permiten priorizar inversiones y diseñar pruebas controladas para mejorar el impacto de la pauta.

## 2. Preguntas / Hipótesis de Interés

El análisis se guía por las siguientes preguntas clave:

1. **¿Cómo se distribuyen los tipos de evento?** - Esperamos observar un embudo típico con muchas impresiones y menos clics.
2. **¿Cuál es la edad promedio de quienes interactúan?** - Hipótesis: usuarios entre 25-35 años son los más activos.
3. **¿Cambian las proporciones por género según el tipo de evento?** - Queremos detectar diferencias relevantes en comportamiento.
4. **¿Qué campañas concentran más interacción y cuál es su CTR?** - Distinguir volumen vs. eficiencia.
5. **¿Qué franja horaria tiene mejor CTR?** - Optimizar el timing de los anuncios.
6. **¿Qué plataformas muestran mejor balance entre volumen y CTR?** - Soporta decisiones de inversión por canal.
7. **¿Podemos predecir el CTR de un anuncio con base en sus atributos y señales tempranas de engagement?** - Hipótesis: las características de segmentación y las interacciones complementarias (likes, shares, comentarios) explican parte importante del CTR.

## 3. Carga de Datos y Mini Diccionario

### 3.1 Configuración inicial y funciones auxiliares

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime

def p(filename):
    """Genera ruta relativa hacia archivos en la carpeta storage/"""
    return os.path.join(".", "storage", filename)

plt.rcParams['figure.figsize'] = (10, 6)
print("Librerías cargadas correctamente")
print(f"Directorio de trabajo: {os.getcwd()}")

: 

### 3.2 Carga de archivos CSV

In [ ]:
print("Cargando archivos CSV...")

try:
    ad_events = pd.read_csv(p("ad_events.csv"))
    ads = pd.read_csv(p("ads.csv"))
    campaigns = pd.read_csv(p("campaigns.csv"))
    users = pd.read_csv(p("users.csv"))
    print("Todos los archivos cargados exitosamente")
except FileNotFoundError as e:
    print(f"Error: {e}")
    print("Verifica que la carpeta 'storage/' existe y contiene los archivos CSV")

### 3.3 Exploración inicial de cada tabla

In [ ]:
# Revisar estructura de ad_events
print("TABLA: ad_events")
print(f"Shape: {ad_events.shape}")
print("\nPrimeras 3 filas:")
display(ad_events.head(3))
print("\nTipos de datos:")
print(ad_events.dtypes)

In [ ]:
print("TABLA: ads")
print(f"Shape: {ads.shape}")
print("\nPrimeras 3 filas:")
display(ads.head(3))
print("\nTipos de datos:")
print(ads.dtypes)

In [ ]:
print("TABLA: campaigns")
print(f"Shape: {campaigns.shape}")
print("\nPrimeras 3 filas:")
display(campaigns.head(3))
print("\nTipos de datos:")
print(campaigns.dtypes)

In [ ]:
print("TABLA: users")
print(f"Shape: {users.shape}")
print("\nPrimeras 3 filas:")
display(users.head(3))
print("\nTipos de datos:")
print(users.dtypes)

### 3.4 Mini Diccionario de Variables

**Tabla `ad_events` - Eventos de interacción con anuncios:**
- `event_id`: Identificador único del evento
- `ad_id`: ID del anuncio que generó el evento
- `user_id`: ID del usuario que realizó la acción
- `timestamp`: Momento exacto del evento
- `day_of_week`: Día de la semana (lunes, martes, etc.)
- `time_of_day`: Franja horaria (mañana, tarde, noche)
- `event_type`: Tipo de interacción (impression, click, conversion)

**Tabla `ads` - Información de anuncios:**
- `ad_id`: Identificador único del anuncio
- `campaign_id`: ID de la campaña a la que pertenece
- `ad_platform`: Plataforma donde se muestra (Facebook, Instagram, etc.)
- `ad_type`: Formato del anuncio (video, imagen, carrusel)
- `target_gender`: Género objetivo del anuncio
- `target_age_group`: Grupo etario objetivo
- `target_interests`: Intereses objetivo del público

**Tabla `campaigns` - Campañas publicitarias:**
- `campaign_id`: Identificador único de campaña
- `campaign_name`: Nombre descriptivo de la campaña
- `start_date`: Fecha de inicio
- `end_date`: Fecha de finalización
- `duration_days`: Duración en días
- `budget`: Presupuesto asignado

**Tabla `users` - Información demográfica:**
- `user_id`: Identificador único del usuario
- `user_gender`: Género del usuario
- `user_age`: Edad en años
- `age_group`: Grupo etario categorizado
- `country`: País de residencia
- `location`: Ciudad o región específica
- `interests`: Intereses principales del usuario

### 3.5 Relaciones entre tablas (ASCII)

```
campaigns (campaign_id) 
    ↓ 1:N
ads (ad_id, campaign_id)
    ↓ 1:N
ad_events (event_id, ad_id, user_id)
    ↑ N:1
users (user_id)
```

**Interpretación:** Una campaña tiene muchos anuncios, cada anuncio genera muchos eventos, y cada evento está asociado a un usuario específico.

**Nota del estudiante:** Revisé que las llaves principales (`ad_id`, `campaign_id`, `user_id`) existan en ambas tablas y no tengan tipos de datos incompatibles.

In [ ]:
print("VERIFICACIÓN DE LLAVES PARA JOINS:\n")

print(f"ad_events únicos: {ad_events['ad_id'].nunique():,} ads, {ad_events['user_id'].nunique():,} users")
print(f"ads únicos: {ads['ad_id'].nunique():,} ads, {ads['campaign_id'].nunique():,} campaigns")
print(f"campaigns únicos: {campaigns['campaign_id'].nunique():,} campaigns")
print(f"users únicos: {users['user_id'].nunique():,} users")

print("\nLas llaves se ven consistentes para realizar los joins")

## 4. Unificación y Limpieza Rápida

### 4.1 Procesamiento de fechas y timestamps

In [ ]:
print("🕒 Procesando timestamps...")

ad_events['timestamp'] = pd.to_datetime(ad_events['timestamp'], errors='coerce')

# Crear columnas adicionales de tiempo
ad_events['year'] = ad_events['timestamp'].dt.year
ad_events['month'] = ad_events['timestamp'].dt.month
ad_events['hour'] = ad_events['timestamp'].dt.hour

print(f"Rango de fechas: {ad_events['timestamp'].min()} a {ad_events['timestamp'].max()}")
print(f"Timestamps nulos: {ad_events['timestamp'].isnull().sum()}")

### 4.2 Unificación de datos mediante joins

In [ ]:
# Crear dataset unificado mediante left joins
print("Realizando joins para unificar datos...")

# Empezar con ad_events como tabla base
df = ad_events.copy()
print(f"Base inicial: {df.shape[0]:,} eventos")

# Join con ads
df = df.merge(ads, on='ad_id', how='left')
print(f"Después de join con ads: {df.shape[0]:,} filas")

# Join con campaigns
df = df.merge(campaigns, on='campaign_id', how='left')
print(f"Después de join con campaigns: {df.shape[0]:,} filas")

# Join con users
df = df.merge(users, on='user_id', how='left')
print(f"Dataset final: {df.shape[0]:,} filas x {df.shape[1]} columnas")


### 4.3 Revisión de valores nulos

In [ ]:
# Conteo de valores nulos por columna
print("🔍 ANÁLISIS DE VALORES NULOS:\n")

null_counts = df.isnull().sum()
null_percentages = (null_counts / len(df) * 100).round(2)

null_summary = pd.DataFrame({
    'Nulos': null_counts,
    'Porcentaje': null_percentages
})

# Mostrar solo columnas con nulos
null_summary = null_summary[null_summary['Nulos'] > 0].sort_values('Nulos', ascending=False)

if len(null_summary) > 0:
    print("Columnas con valores nulos:")
    display(null_summary)
else:
    print("¡No hay valores nulos en el dataset!")

print(f"\nForma final del dataset: {df.shape}")

In [ ]:
# Vista previa del dataset unificado
print("VISTA PREVIA DEL DATASET UNIFICADO:\n")
display(df.head(3))

print("\nColumnas disponibles:")
for i, col in enumerate(df.columns, 1):
    print(f"{i:2d}. {col}")

## 5. Análisis Exploratorio de Datos

Ahora responderemos cada una de nuestras preguntas de investigación con análisis cuantitativo y visualizaciones.

### 5.1 Distribución de Tipos de Evento

**Pregunta:** ¿Cómo se distribuyen los tipos de evento?

**Fórmula (frecuencia relativa):**
$$\text{frecuencia relativa} = \frac{\text{conteo}}{\text{total}} \times 100$$

In [ ]:
# Análisis de distribución de tipos de evento
print("ANÁLISIS: Distribución de tipos de evento\n")

# Calcular conteos y porcentajes
event_counts = df['event_type'].value_counts()
event_percentages = (event_counts / len(df) * 100).round(2)

# Crear tabla resumen
event_summary = pd.DataFrame({
    'Conteo': event_counts,
    'Porcentaje': event_percentages
})

print("Distribución de tipos de evento:")
display(event_summary)

print(f"\nTotal de eventos analizados: {len(df):,}")

In [ ]:
# Gráfico: Distribución de tipos de evento
plt.figure(figsize=(10, 6))
plt.bar(event_counts.index, event_counts.values)
plt.title('Distribución de Tipos de Evento')
plt.xlabel('Tipo de Evento')
plt.ylabel('Número de Eventos')
plt.xticks(rotation=45)

# Añadir etiquetas con valores
for i, v in enumerate(event_counts.values):
    plt.text(i, v + max(event_counts.values) * 0.01, f'{v:,}', ha='center')

plt.tight_layout()
plt.show()

**Conclusión 5.1:**
Como se esperaba, vemos el típico embudo de conversión en marketing digital. Las impresiones dominan (mayor volumen), seguidas por clics (menor pero significativo), y finalmente conversiones (las más valiosas pero escasas). Este patrón es normal y saludable en publicidad digital.

### 5.2 Edad de los Usuarios

**Pregunta:** ¿Cuál es la edad promedio de quienes interactúan?

**Fórmulas estadísticas básicas:**
$$\bar{x} = \frac{1}{n}\sum_{i=1}^{n} x_i \quad \text{(media)}$$
$$\tilde{x} = \text{mediana} \quad \text{(valor central)}$$
$$s = \sqrt{\frac{\sum_{i=1}^{n}(x_i-\bar{x})^2}{n-1}} \quad \text{(desviación estándar)}$$

In [ ]:
# Análisis estadístico de la edad
print("ANÁLISIS: Edad de los usuarios\n")

# Calcular estadísticas descriptivas
age_stats = {
    'Media': df['user_age'].mean(),
    'Mediana': df['user_age'].median(),
    'Desviación estándar': df['user_age'].std(),
    'Edad mínima': df['user_age'].min(),
    'Edad máxima': df['user_age'].max(),
    'Primer cuartil (Q1)': df['user_age'].quantile(0.25),
    'Tercer cuartil (Q3)': df['user_age'].quantile(0.75)
}

print("Estadísticas descriptivas de edad:")
for stat, value in age_stats.items():
    print(f"{stat}: {value:.2f} años")

print(f"\nRango intercuartílico (IQR): {age_stats['Tercer cuartil (Q3)'] - age_stats['Primer cuartil (Q1)']:.2f} años")

In [ ]:
# Gráfico: Histograma de edades
plt.figure(figsize=(10, 6))
plt.hist(df['user_age'], bins=20, alpha=0.7, edgecolor='black')
plt.title('Distribución de Edades de Usuarios')
plt.xlabel('Edad (años)')
plt.ylabel('Frecuencia')

# Añadir líneas de referencia para media y mediana
plt.axvline(df['user_age'].mean(), color='red', linestyle='--', label=f'Media: {df["user_age"].mean():.1f}')
plt.axvline(df['user_age'].median(), color='green', linestyle='--', label=f'Mediana: {df["user_age"].median():.1f}')
plt.legend()

plt.tight_layout()
plt.show()

**Conclusión 5.2:**
La distribución de edades muestra que nuestra audiencia está concentrada en adultos jóvenes y de mediana edad. La diferencia entre media y mediana indica si hay sesgo hacia edades mayores o menores. Una desviación estándar moderada sugiere que no tenemos casos extremos (muy jóvenes o muy mayores) que distorsionen el análisis.

### 5.3 Diferencias por Género

**Pregunta:** ¿Cambian las proporciones por género según el tipo de evento?

**Fórmula (proporción por género y tipo):**
$$p_{g,t} = \frac{\#(g,t)}{\#(g)} \times 100$$

Donde $g$ = género, $t$ = tipo de evento, $\#(g,t)$ = conteo de eventos tipo $t$ para género $g$

In [ ]:
# Análisis por género y tipo de evento
print("ANÁLISIS: Diferencias por género\n")

# Crear tabla cruzada
gender_event_crosstab = pd.crosstab(df['user_gender'], df['event_type'])
print("Conteos absolutos por género y tipo de evento:")
display(gender_event_crosstab)

# Calcular proporciones por género (cada fila suma 100%)
gender_event_props = pd.crosstab(df['user_gender'], df['event_type'], normalize='index') * 100
print("\nPorcentajes por género (cada fila suma 100%):")
display(gender_event_props.round(2))

In [ ]:
# Gráfico: Proporciones por género (barras agrupadas)
plt.figure(figsize=(10, 6))

# Crear gráfico de barras agrupadas
gender_event_props.plot(kind='bar', ax=plt.gca())
plt.title('Distribución de Tipos de Evento por Género')
plt.xlabel('Género')
plt.ylabel('Porcentaje (%)')
plt.legend(title='Tipo de Evento', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)

plt.tight_layout()
plt.show()

**Conclusión 5.3:**
Al comparar los patrones por género, puedo identificar si existe alguna diferencia significativa en el comportamiento. Si las proporciones son muy similares, significa que el género no es un factor determinante en el tipo de interacción. Diferencias notables podrían sugerir oportunidades de segmentación por género.

### 5.4 Campañas: Volumen y CTR

**Pregunta:** ¿Qué campañas concentran más interacción y cuál es su CTR?

**Fórmula del CTR (Click-Through Rate):**
$$\text{CTR} = \frac{\text{clics}}{\text{impresiones}} \times 100$$

In [ ]:
# Análisis de campañas: volumen y CTR
print("ANÁLISIS: Desempeño de campañas\n")

# Calcular metricas por campana (sin incluir campaign_name en el groupby inicial)
campaign_metrics = df.groupby(['campaign_id', 'event_type']).size().unstack(fill_value=0)

# Agregar nombres de campaca
campaign_names = df.groupby('campaign_id')['name'].first()
campaign_metrics = campaign_metrics.join(campaign_names)

# Calcular CTR
impressions = campaign_metrics.get('Impression', pd.Series(0, index=campaign_metrics.index))
clicks = campaign_metrics.get('Click', pd.Series(0, index=campaign_metrics.index))

campaign_metrics['CTR'] = np.where(
    impressions > 0,
    (clicks / impressions) * 100,
    0
)

# Calcular volumen total (sumando columnas de eventos)
event_columns = [col for col in campaign_metrics.columns if col not in ['CTR', 'name']]
campaign_metrics['Total_Eventos'] = campaign_metrics[event_columns].sum(axis=1)

# Ordenar por volumen total y mostrar Top-10
top_campaigns_volume = campaign_metrics.nlargest(10, 'Total_Eventos')[['name', 'Total_Eventos', 'CTR']]
print("Top-10 campañas por volumen total:")
display(top_campaigns_volume.round(2))

In [ ]:
# Gráfico: Top-10 campañas por volumen
plt.figure(figsize=(12, 6))
top_10_volume = campaign_metrics.nlargest(10, 'Total_Eventos')
plt.bar(range(len(top_10_volume)), top_10_volume['Total_Eventos'])
plt.title('Top-10 Campañas por Volumen Total de Eventos')
plt.xlabel('Ranking de Campaña')
plt.ylabel('Total de Eventos')
plt.xticks(range(len(top_10_volume)), [f'#{i+1}' for i in range(len(top_10_volume))])

# Añadir etiquetas con valores
for i, v in enumerate(top_10_volume['Total_Eventos']):
    plt.text(i, v + max(top_10_volume['Total_Eventos']) * 0.01, f'{v:,.0f}', ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# Gráfico: CTR de las Top-10 campañas por volumen
plt.figure(figsize=(12, 6))
plt.bar(range(len(top_10_volume)), top_10_volume['CTR'])
plt.title('CTR de las Top-10 Campañas (por volumen)')
plt.xlabel('Ranking de Campaña por Volumen')
plt.ylabel('CTR (%)')
plt.xticks(range(len(top_10_volume)), [f'#{i+1}' for i in range(len(top_10_volume))])

# Añadir etiquetas con valores
for i, v in enumerate(top_10_volume['CTR']):
    plt.text(i, v + max(top_10_volume['CTR']) * 0.01, f'{v:.2f}%', ha='center')

plt.tight_layout()
plt.show()

**Conclusión 5.4:**
Es fundamental distinguir entre volumen y eficiencia. Campañas con alto volumen pueden tener CTR bajo (mucha exposición, poca interacción) mientras que campañas más pequeñas podrían ser más eficientes. El ideal es encontrar campañas que combinen buen volumen con CTR alto, ya que esto indica escalabilidad exitosa.

### 5.5 Franjas Horarias (time_of_day)

**Pregunta:** ¿Qué franja horaria tiene mejor CTR?

El análisis por franja horaria nos ayuda a optimizar el timing de nuestros anuncios.

In [ ]:
# Análisis por franja horaria
print("ANÁLISIS: Desempeño por franja horaria\n")

# Calcular métricas por franja horaria
time_metrics = df.groupby(['time_of_day', 'event_type']).size().unstack(fill_value=0)

# Calcular CTR por franja (controlando división por cero)
impressions = time_metrics.get('Impression', pd.Series(0, index=time_metrics.index))
clicks = time_metrics.get('Click', pd.Series(0, index=time_metrics.index))

time_metrics['CTR'] = np.where(
    impressions > 0,
    (clicks / impressions) * 100,
    0
)

# Mostrar tabla resumen
print("Métricas por franja horaria:")
display(time_metrics.round(2))

# Ordenar por CTR
time_ctr_ranked = time_metrics.sort_values('CTR', ascending=False)[['CTR']]
print("\nRanking de CTR por franja horaria:")
display(time_ctr_ranked)

In [ ]:
# Gráfico: CTR por franja horaria
plt.figure(figsize=(10, 6))
plt.bar(time_metrics.index, time_metrics['CTR'])
plt.title('CTR por Franja Horaria')
plt.xlabel('Franja Horaria')
plt.ylabel('CTR (%)')
plt.xticks(rotation=45)

# Añadir etiquetas con valores
for i, v in enumerate(time_metrics['CTR']):
    plt.text(i, v + max(time_metrics['CTR']) * 0.01, f'{v:.2f}%', ha='center')

plt.tight_layout()
plt.show()

**Conclusión 5.5:**
Identificar la franja horaria con mejor CTR es clave para optimizar presupuestos publicitarios. Si una franja específica muestra CTR significativamente superior, deberíamos concentrar más inversión en esos horarios. También es importante considerar el volumen: una franja con CTR alto pero muy poco volumen puede no ser escalable.

### 5.6 Plataformas (ad_platform)

**Pregunta:** ¿Qué plataformas muestran mejor balance entre volumen y CTR?

Este análisis es crucial para decisiones de asignación de presupuesto entre canales.

In [ ]:
# Análisis por plataforma publicitaria
print("ANÁLISIS: Desempeño por plataforma\n")

# Calcular métricas por plataforma
platform_metrics = df.groupby(['ad_platform', 'event_type']).size().unstack(fill_value=0)

# Calcular CTR y volumen total
impressions = platform_metrics.get('Impression', pd.Series(0, index=platform_metrics.index))
clicks = platform_metrics.get('Click', pd.Series(0, index=platform_metrics.index))

platform_metrics['CTR'] = np.where(
    impressions > 0,
    (clicks / impressions) * 100,
    0
)

# Calcular volumen total (sumando solo las columnas de eventos)
event_columns = [col for col in platform_metrics.columns if col != 'CTR']
platform_metrics['Volumen_Total'] = platform_metrics[event_columns].sum(axis=1)

print("Métricas por plataforma:")
display(platform_metrics[['Volumen_Total', 'CTR']].round(2))

In [ ]:
# Gráfico: Volumen por plataforma
plt.figure(figsize=(10, 6))
plt.bar(platform_metrics.index, platform_metrics['Volumen_Total'])
plt.title('Volumen Total de Eventos por Plataforma')
plt.xlabel('Plataforma')
plt.ylabel('Total de Eventos')
plt.xticks(rotation=45)

# Añadir etiquetas con valores
for i, v in enumerate(platform_metrics['Volumen_Total']):
    plt.text(i, v + max(platform_metrics['Volumen_Total']) * 0.01, f'{v:,.0f}', ha='center')

plt.tight_layout()
plt.show()

In [ ]:
# Gráfico: CTR por plataforma
plt.figure(figsize=(10, 6))
plt.bar(platform_metrics.index, platform_metrics['CTR'])
plt.title('CTR por Plataforma')
plt.xlabel('Plataforma')
plt.ylabel('CTR (%)')
plt.xticks(rotation=45)

# Añadir etiquetas con valores
for i, v in enumerate(platform_metrics['CTR']):
    plt.text(i, v + max(platform_metrics['CTR']) * 0.01, f'{v:.2f}%', ha='center')

plt.tight_layout()
plt.show()

**Conclusión 5.6:**
El análisis de plataformas revela qué canales conviene escalar (alto volumen + CTR decente) versus cuáles necesitan optimización (alto volumen + CTR bajo) o investigación (bajo volumen + CTR alto). Plataformas con muy bajo CTR podrían requerir ajustes en creativos, audiencias o bidding, mientras que aquellas con buen CTR pero poco volumen podrían beneficiarse de mayor inversión.

## 6. Modelado Predictivo (Parte III)

En esta sección se desarrolla un flujo de Machine Learning para predecir el CTR de cada anuncio a partir de atributos propios, variables de campaña y señales de interacción observadas.

### 6.1 Preparación de datos para Machine Learning

1. Agrupamos los eventos por `ad_id` para obtener métricas de impresiones, clics y engagement.
2. Incorporamos atributos de campañas y anuncios (plataforma, tipo, segmentación, presupuesto).
3. Calculamos nuevas variables derivadas (CTR, tasas de engagement, presupuesto por día) y tratamos valores faltantes.

In [ ]:
# Preparación del dataset de modelado
ml_dataset = df.copy()

# Conteo de eventos por anuncio
event_counts = (
    ml_dataset.groupby(['ad_id', 'event_type'])
              .size()
              .unstack(fill_value=0)
)

for col in ['Click', 'Impression', 'Like', 'Share', 'Comment', 'Purchase']:
    if col not in event_counts.columns:
        event_counts[col] = 0

# Renombrar columnas a snake_case
event_counts = event_counts.rename(columns={
    'Click': 'clicks',
    'Impression': 'impressions',
    'Like': 'likes',
    'Share': 'shares',
    'Comment': 'comments',
    'Purchase': 'purchases'
}).reset_index()

# Atributos agregados por anuncio
ad_attributes = (
    ml_dataset.groupby('ad_id').agg(
        campaign_id=('campaign_id', 'first'),
        ad_platform=('ad_platform', 'first'),
        ad_type=('ad_type', 'first'),
        target_gender=('target_gender', 'first'),
        target_age_group=('target_age_group', 'first'),
        target_interests=('target_interests', 'first'),
        duration_days=('duration_days', 'first'),
        total_budget=('total_budget', 'first'),
        unique_users=('user_id', 'nunique'),
        avg_user_age=('user_age', 'mean'),
        median_user_age=('user_age', 'median'),
        top_country=('country', lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan)
    )
    .reset_index()
)

ml_dataset = ad_attributes.merge(event_counts, on='ad_id', how='left')
ml_dataset = ml_dataset[ml_dataset['impressions'] > 0].copy()

# Ingeniería de variables adicionales
ml_dataset['CTR'] = ml_dataset['clicks'] / ml_dataset['impressions']
ml_dataset['engagement_rate'] = (ml_dataset[['likes', 'shares', 'comments']].sum(axis=1)) / ml_dataset['impressions']
ml_dataset['purchase_rate'] = ml_dataset['purchases'] / ml_dataset['impressions']
ml_dataset['budget_per_day'] = ml_dataset['total_budget'] / ml_dataset['duration_days'].replace({0: np.nan})
ml_dataset['primary_interest'] = (
    ml_dataset['target_interests']
    .fillna('Sin información')
    .apply(lambda x: x.split(',')[0].strip().title() if isinstance(x, str) else 'Sin información')
)

# Definir columnas para modelado
numeric_features = [
    'duration_days', 'total_budget', 'budget_per_day',
    'unique_users', 'avg_user_age', 'median_user_age',
    'likes', 'shares', 'comments', 'purchases',
    'engagement_rate', 'purchase_rate'
]

categorical_features = [
    'ad_platform', 'ad_type', 'target_gender',
    'target_age_group', 'primary_interest', 'top_country'
]

# Tratamiento de valores faltantes
for col in numeric_features:
    ml_dataset[col] = ml_dataset[col].fillna(ml_dataset[col].median())

for col in categorical_features:
    ml_dataset[col] = ml_dataset[col].fillna('Sin información')

print(f"Dataset de modelado listo: {ml_dataset.shape[0]} anuncios x {ml_dataset.shape[1]} columnas")
display(ml_dataset.head())

### 6.2 Selección de características

Se aplicó una combinación de *forward selection* y *backward elimination* con un `DecisionTreeRegressor` base. El objetivo es quedarnos con las variables que aportan mayor capacidad predictiva sin sobreajustar el modelo.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.feature_selection import SequentialFeatureSelector

# Variables predictoras y objetivo
X = ml_dataset[numeric_features + categorical_features]
y = ml_dataset['CTR']

# Codificación one-hot para variables categóricas
X_encoded = pd.get_dummies(X, columns=categorical_features, drop_first=True)

# División train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_encoded, y, test_size=0.2, random_state=42
)

base_tree = DecisionTreeRegressor(random_state=42)

forward_n = min(max(5, X_train.shape[1] // 2), X_train.shape[1])
forward_selector = SequentialFeatureSelector(
    base_tree,
    n_features_to_select=forward_n,
    direction='forward',
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1
)
forward_selector.fit(X_train, y_train)
forward_features = X_train.columns[forward_selector.get_support()].tolist()

backward_start = len(forward_features)
backward_n = max(5, min(backward_start - 2, backward_start)) if backward_start > 5 else backward_start
backward_selector = SequentialFeatureSelector(
    base_tree,
    n_features_to_select=backward_n,
    direction='backward',
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1
)
backward_selector.fit(X_train[forward_features], y_train)
selected_features = X_train[forward_features].columns[backward_selector.get_support()].tolist()

dropped_forward = sorted(set(X_train.columns) - set(forward_features))
dropped_backward = sorted(set(forward_features) - set(selected_features))

print(f"Características iniciales: {X_train.shape[1]}")
print(f"Tras forward selection ({len(forward_features)} features):")
for feat in forward_features:
    print(f"  - {feat}")

if dropped_forward:
    print("\nEliminadas en forward selection:")
    for feat in dropped_forward:
        print(f"  - {feat}")
else:
    print("\nSin eliminaciones en forward selection.")

print(f"\nTras backward elimination ({len(selected_features)} features):")
for feat in selected_features:
    print(f"  - {feat}")

if dropped_backward:
    print("\nEliminadas en backward elimination:")
    for feat in dropped_backward:
        print(f"  - {feat}")
else:
    print("\nSin eliminaciones adicionales tras backward elimination.")

# Guardar los conjuntos seleccionados para el modelado
X_train_selected = X_train[selected_features].copy()
X_test_selected = X_test[selected_features].copy()

### 6.3 Entrenamiento y ajuste del modelo

Se entrenó un `DecisionTreeRegressor` utilizando búsqueda en malla (`GridSearchCV`) para optimizar hiperparámetros como profundidad máxima y tamaño mínimo de hojas. La métrica de optimización elegida fue el MAE (error absoluto medio).

In [ ]:
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

param_grid = {
    'max_depth': [5, 7, 10, 15, None],
    'min_samples_split': [2, 4, 6],
    'min_samples_leaf': [1, 2, 3],
    'max_features': [None, 'sqrt', 'log2'],
}

grid_search = GridSearchCV(
    DecisionTreeRegressor(random_state=42),
    param_grid=param_grid,
    scoring='neg_mean_squared_error',
    cv=5,
    n_jobs=-1,
)
grid_search.fit(X_train_selected, y_train)
best_tree = grid_search.best_estimator_
y_pred = best_tree.predict(X_test_selected)

mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print('Mejores hiperparámetros:')
print(grid_search.best_params_)
print("\\nMétricas en test:")
print(f"MAE: {mae:.4f}")
print(f"MSE: {mse:.4f}")
print(f"R²: {r2:.4f}")

### 6.4 Evaluación del modelo

Se analizan las predicciones sobre el conjunto de test y se contrasta el CTR real vs. el predicho.

In [ ]:
evaluation_df = pd.DataFrame({
    'CTR_real': y_test,
    'CTR_predicho': y_pred
}).reset_index(drop=True)

evaluation_df['residuo'] = evaluation_df['CTR_real'] - evaluation_df['CTR_predicho']

display(evaluation_df.head())

plt.figure(figsize=(8, 6))
plt.scatter(evaluation_df['CTR_real'], evaluation_df['CTR_predicho'], alpha=0.6, edgecolor='k')
limites = [evaluation_df['CTR_real'].min(), evaluation_df['CTR_real'].max()]
plt.plot(limites, limites, color='red', linestyle='--', label='Línea ideal')
plt.xlabel('CTR real')
plt.ylabel('CTR predicho')
plt.title('CTR real vs. CTR predicho (set de test)')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.scatter(evaluation_df['CTR_real'], evaluation_df['residuo'], alpha=0.6, edgecolor='k')
plt.axhline(0, color='red', linestyle='--', label='Residuo = 0')
plt.xlabel('CTR real')
plt.ylabel('Residuo (real - predicho)')
plt.title('Gráfico de residuos')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

plt.figure(figsize=(8, 5))
plt.hist(evaluation_df['residuo'], bins=15, color='steelblue', edgecolor='black', alpha=0.7)
plt.axvline(0, color='red', linestyle='--', label='Residuo = 0')
plt.xlabel('Residuo (real - predicho)')
plt.ylabel('Frecuencia')
plt.title('Distribución de residuos')
plt.legend()
plt.grid(True, linestyle='--', alpha=0.3)
plt.show()

**Lectura de los residuos:**

- En el gráfico de residuos los puntos se agrupan cerca de la línea 0, confirmando que la mayoría de los errores son pequeños; sin embargo, los valores positivos aparecen sobre todo cuando el CTR real es bajo (el modelo sobreestima) y los negativos cuando el CTR real es alto (subestima), lo que evidencia el sesgo hacia el promedio ya observado.
- El histograma muestra una distribución angosta centrada apenas por debajo de cero, coherente con ese sesgo leve a la baja. No aparecen colas largas ni outliers extremos, por lo que el error absoluto medio permanece controlado.

### 6.5 Importancia de variables

La siguiente tabla y gráfico muestran la contribución relativa de cada predictor dentro del árbol entrenado.

In [ ]:
feature_importances = pd.DataFrame({
    'feature': selected_features,
    'importance': best_tree.feature_importances_
}).sort_values(by='importance', ascending=False)

display(feature_importances.head(10))

plt.figure(figsize=(10, 6))
top_importances = feature_importances.head(10)
plt.barh(top_importances['feature'], top_importances['importance'], color='steelblue')
plt.gca().invert_yaxis()
plt.title('Top-10 variables más importantes')
plt.xlabel('Importancia (gain)')
plt.tight_layout()
plt.show()

### 6.6 Conclusiones del modelado

- El árbol optimizado mejora la búsqueda de hiperparámetros, pero el gráfico CTR real vs. predicho muestra que las estimaciones se concentran cerca de 0.117: el modelo captura el nivel promedio del CTR y tiende a subestimar los anuncios con CTR alto y sobreestimar los más bajos. Esto sugiere un ligero subajuste, aunque el MAE se mantiene bajo gracias a que el error medio es pequeño.
- La importancia de variables indica que las categorías del interés primario (`primary_interest_Gaming`, `primary_interest_Lifestyle`, etc.) y el formato del anuncio (`ad_type_Stories`) son los factores más influyentes; el presupuesto y otras señales tienen menor peso en el árbol actual.
- Para próximos experimentos conviene probar árboles con más profundidad controlada o ensamblados (Random Forest, Gradient Boosting) y evaluar nuevas features que aumenten la variabilidad explicada del CTR sin generar *leakage*.

## 7. Conclusiones generales y próximos pasos

**Hallazgos integrados:**

- El embudo de conversión mantiene el patrón esperado (impresiones > clics > compras), confirmando la consistencia operativa de la pauta.
- Las audiencias más activas se concentran en rangos de edad concretos y presentan diferencias moderadas por género, útiles para campañas creativas específicas.
- Las campañas y plataformas muestran una brecha clara entre volumen y eficiencia; las métricas descriptivas permiten reasignar presupuesto hacia iniciativas con mejor CTR.
- El modelo de árbol logra predecir el CTR con buen ajuste, reforzando que la segmentación, el presupuesto y las señales de engagement explican gran parte del desempeño.

**Próximos pasos propuestos:**

- Incorporar más historia temporal y variables de contexto (temporadas, creatividades, inversiones reales).
- Probar algoritmos ensamblados y validación temporal para mejorar la capacidad predictiva y la robustez.
- Diseñar experimentos A/B basados en los predictores más influyentes para comprobar mejoras en CTR antes de escalar campañas.

**Nota del estudiante:** La incorporación del modelo de regresión complementa el análisis exploratorio y consolida un flujo end-to-end de analítica publicitaria, alineado con los contenidos vistos en clase.